Custom Model Developing using decomposition and multiconstraint optimization

In [24]:
import pandas as pd

df=pd.read_csv("../data/processed/bkk_cleaned.csv", low_memory=False)
print("\n--- 1. Data Type & Null Check ---")
print(df.info())

print("\n--- 2. Missing Values Count ---")
print(df.isnull().sum())

print("\n--- 3. Outlier Detection (The Ghost bus check) ---")
cols_to_check = ['seconds_to_next_stop', 'vehicle.currentStopSequence', 'vehicle.position.bearing']
existing_cols = [col for col in cols_to_check if col in df.columns]
print(df[existing_cols].describe())



--- 1. Data Type & Null Check ---
<class 'pandas.DataFrame'>
RangeIndex: 193555 entries, 0 to 193554
Data columns (total 20 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ingested_at                        193555 non-null  str    
 1   id                                 193555 non-null  str    
 2   vehicle.vehicle.id                 193555 non-null  str    
 3   vehicle.vehicle.label              193555 non-null  str    
 4   vehicle.vehicle.licensePlate       193017 non-null  str    
 5   vehicle.position.bearing           192973 non-null  float64
 6   vehicle.position.latitude          193555 non-null  float64
 7   vehicle.position.longitude         193555 non-null  float64
 8   vehicle.timestamp                  193555 non-null  float64
 9   vehicle.currentStatus              193555 non-null  str    
 10  vehicle.position.speed             193499 non-null  str    
 11  vehicle.trip.tr

Columns Name remapping

In [25]:
rename_mapping = {
    'vehicle.vehicle.id': 'vehicle_id',
    'vehicle.vehicle.label': 'route_label',
    'vehicle.vehicle.licensePlate': 'license_plate',
    'vehicle.position.bearing': 'bearing',
    'vehicle.position.latitude': 'latitude',
    'vehicle.position.longitude': 'longitude',
    'vehicle.timestamp': 'timestamp',
    'vehicle.currentStatus': 'current_status',
    'vehicle.position.speed': 'speed',
    'vehicle.trip.tripId': 'trip_id',
    'vehicle.trip.routeId': 'route_id',
    'vehicle.trip.startDate': 'start_date',
    'vehicle.trip.scheduleRelationship': 'schedule_relationship',
    'vehicle.currentStopSequence': 'current_stop_sequence',
    'vehicle.congestionLevel': 'congestion_level'
}

df = df.rename(columns=rename_mapping)

print("Columns successfully renamed!")
print(df.info())

Columns successfully renamed!
<class 'pandas.DataFrame'>
RangeIndex: 193555 entries, 0 to 193554
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ingested_at            193555 non-null  str    
 1   id                     193555 non-null  str    
 2   vehicle_id             193555 non-null  str    
 3   route_label            193555 non-null  str    
 4   license_plate          193017 non-null  str    
 5   bearing                192973 non-null  float64
 6   latitude               193555 non-null  float64
 7   longitude              193555 non-null  float64
 8   timestamp              193555 non-null  float64
 9   current_status         193555 non-null  str    
 10  speed                  193499 non-null  str    
 11  trip_id                193555 non-null  str    
 12  route_id               193555 non-null  str    
 13  start_date             193555 non-null  int64  
 14  schedule_relation

Final Data cleaning

In [26]:
print(f"Original shape: {df.shape}")

if 'vehicle.stopId' in df.columns:
    df = df.drop(columns=['vehicle.stopId'])

df['current_stop_sequence'] = pd.to_numeric(df['current_stop_sequence'], errors='coerce')

df['congestion_level'] = df['congestion_level'].fillna('UNKNOWN')

critical_cols = [
    "current_stop_sequence",
    "bearing",
    "speed",
    "license_plate"
]
df = df.dropna(subset=critical_cols)

print(f"Cleaned shape: {df.shape}")
print("\nFinal Null Count:")
print(df.isnull().sum())


Original shape: (193555, 20)
Cleaned shape: (92809, 19)

Final Null Count:
ingested_at              0
id                       0
vehicle_id               0
route_label              0
license_plate            0
bearing                  0
latitude                 0
longitude                0
timestamp                0
current_status           0
speed                    0
trip_id                  0
route_id                 0
start_date               0
schedule_relationship    0
current_stop_sequence    0
congestion_level         0
arrival_timestamp        0
seconds_to_next_stop     0
dtype: int64


Data Converting, Normalization and introduction of some dimensions before ML pipeline

In [27]:
import numpy as np

print(f"Original shape before engineering: {df.shape}")

# 1. CYCLICAL ENCODING
# Convert Unix timestamp to a pandas datetime object
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

df['hour_continuous'] = df['datetime'].dt.hour + (df['datetime'].dt.minute / 60)

# Map the 24-hour cycle onto unit circle
df['hour_sin'] = np.sin(2 * np.pi * df['hour_continuous'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_continuous'] / 24)

# Map the 360-degree compass bearing onto unit circle
df['bearing_sin'] = np.sin(2 * np.pi * df['bearing'] / 360)
df['bearing_cos'] = np.cos(2 * np.pi * df['bearing'] / 360)

# 2. CATEGORICAL ENCODING (Low Cardinality)
df = pd.get_dummies(
    df,
    columns=['current_status', 'congestion_level'],
    prefix=['status', 'traffic'],
    drop_first=False,
)

# 3. CATEGORICAL ENCODING (High Cardinality)
route_freq = df['route_id'].value_counts(normalize=True)
df['route_freq'] = df['route_id'].map(route_freq)

# 4. CLEANUP
columns_to_drop = [
    'timestamp', 'datetime', 'hour_continuous', 'bearing', 
    'route_id', 'route_label', 'trip_id', 'start_date', 
    'license_plate', 'schedule_relationship', 'arrival_timestamp'
]

cols_to_drop_safe = [col for col in columns_to_drop if col in df.columns]
df_ml = df.drop(columns=cols_to_drop_safe)

bool_cols = df_ml.select_dtypes(include='bool').columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

print(f"Engineered shape ready for ML: {df_ml.shape}")
print("\nFinal ML Features:")
print(df_ml.columns.tolist())

Original shape before engineering: (92809, 19)
Engineered shape ready for ML: (92809, 71)

Final ML Features:
['ingested_at', 'id', 'vehicle_id', 'latitude', 'longitude', 'speed', 'current_stop_sequence', 'seconds_to_next_stop', 'hour_sin', 'hour_cos', 'bearing_sin', 'bearing_cos', 'status_IN_TRANSIT_TO', 'status_STOPPED_AT', 'traffic_0.0', 'traffic_1.0', 'traffic_10.0', 'traffic_11.0', 'traffic_12.0', 'traffic_13.0', 'traffic_14.0', 'traffic_15.0', 'traffic_16.0', 'traffic_17.0', 'traffic_18.0', 'traffic_19.0', 'traffic_2.0', 'traffic_20.0', 'traffic_21.0', 'traffic_22.0', 'traffic_23.0', 'traffic_24.0', 'traffic_25.0', 'traffic_26.0', 'traffic_27.0', 'traffic_28.0', 'traffic_29.0', 'traffic_3.0', 'traffic_30.0', 'traffic_31.0', 'traffic_32.0', 'traffic_33.0', 'traffic_34.0', 'traffic_35.0', 'traffic_36.0', 'traffic_37.0', 'traffic_38.0', 'traffic_39.0', 'traffic_4.0', 'traffic_40.0', 'traffic_41.0', 'traffic_42.0', 'traffic_43.0', 'traffic_44.0', 'traffic_45.0', 'traffic_46.0', 'traf

In [28]:
df.head()

,ingested_at,id,vehicle_id,route_label,license_plate,bearing,latitude,longitude,timestamp,speed,...,traffic_54.0,traffic_55.0,traffic_56.0,traffic_6.0,traffic_7.0,traffic_8.0,traffic_9.0,traffic_CONGESTION,traffic_UNKNOWN,route_freq
0,2026-04-19 14:11:15.411750,VehiclePosition-BKK_918061825686,918061825686,Budapest-Nyugati,182 568,42.0,48.11273,21.40678,1.776608e+09,0.0,...,False,False,False,False,False,False,False,False,True,0.034124
1,2026-04-19 14:11:15.443021,VehiclePosition-BKK_918061932318,918061932318,Budapest-Keleti,918061932318,76.0,47.18600,20.50780,1.776608e+09,33.055557,...,False,False,False,False,False,False,False,False,True,0.034124
2,2026-04-19 14:11:15.886496,VehiclePosition-BKK_945518150384,945518150384,Budapest-Nyugati,815 038,155.0,47.95259,21.70093,1.776608e+09,10.277778,...,False,False,False,False,False,False,False,False,True,0.009643
3,2026-04-19 14:11:15.488717,VehiclePosition-BKK_918071936382,918071936382,Koprivnica,918071936382,239.0,47.19571,18.60004,1.776608e+09,33.055557,...,False,False,False,False,False,False,False,False,True,0.034124
4,2026-04-19 14:11:15.459819,VehiclePosition-BKK_918061939487,918061939487,Záhony,918061939487,134.0,47.32562,19.47241,1.776608e+09,31.944445,...,False,False,False,False,False,False,False,False,True,0.034124


Base-line Checking

In [29]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np

print("Applying final data type patches...")

# 1. THE FIX: Force speed to be a numeric float
df_ml['speed'] = pd.to_numeric(df_ml['speed'], errors='coerce')

# Drop any rows where the speed was corrupted text and turned into a NaN
df_ml = df_ml.dropna(subset=['speed'])

print("Preparing data for modeling...")

# 2. Define Target (y) and Features (X)
y = df_ml['seconds_to_next_stop']

# Drop the target and the remaining text string identifiers
X = df_ml.drop(columns=['seconds_to_next_stop', 'ingested_at', 'id', 'vehicle_id'])

# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} rows...")
print(f"Testing on {X_test.shape[0]} rows...")

# 4. Initialize and Train the XGBoost Regressor
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1 # Uses all CPU cores
)

print("\nTraining XGBoost Model (This may take a few seconds)...")
model.fit(X_train, y_train)

# 5. Make Predictions on the hidden Test Set
predictions = model.predict(X_test)

# 6. Evaluate the Model's Physics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\n--- BASELINE MODEL RESULTS ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} seconds")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} seconds")

# Convert MAE to minutes for human readability
print(f"\nOn average, the model's delay prediction is off by {mae/60:.2f} minutes.")

Applying final data type patches...
Preparing data for modeling...
Training on 18149 rows...
Testing on 4538 rows...

Training XGBoost Model (This may take a few seconds)...

--- BASELINE MODEL RESULTS ---
Mean Absolute Error (MAE): 86.18 seconds
Root Mean Squared Error (RMSE): 183.57 seconds

On average, the model's delay prediction is off by 1.44 minutes.


Custom ML engine Developing

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

print("Initializing PyTorch Architecture")


class TransitDataset(Dataset):
    def __init__(self, x, y, distance_feature):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

        self.d = torch.tensor(distance_feature.values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.d[idx]
    

class PhysicsInformedNN(nn.Module):
    def __init__(self, input_dim):
        super(PhysicsInformedNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)
    
print("Architecture defined successfully.")

Initializing PyTorch Architecture
Architecture defined successfully.


Custom Training Loop

In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [39]:
# Redefine the training loop with the tuned Lambda
def train_tuned_model(X_scaled, y_train, distance_train):
    v_max = 15.0       
    lambda_pen = 10.0  # LOWERED FROM 100: The 'Goldilocks' penalty
    epochs = 200       # Slightly more epochs to allow for smooth convergence
    batch_size = 64
    
    # Convert to tensors
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    d_tensor = torch.tensor(distance_train, dtype=torch.float32).view(-1, 1)
    
    dataset = TensorDataset(X_tensor, y_tensor, d_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = PhysicsInformedNN(input_dim=X_scaled.shape[1])
    # Lower learning rate slightly for more precise gradient descent
    optimizer = optim.Adam(model.parameters(), lr=0.0005) 
    mse_loss_fn = nn.MSELoss()
    
    print(f"Starting Training with Tuned Lambda: {lambda_pen}...\n")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_X, batch_y, batch_d in dataloader:
            optimizer.zero_grad()
            y_pred = model(batch_X)
            
            base_mse = mse_loss_fn(y_pred, batch_y)
            
            # The KKT Constraint Math
            min_possible_time = batch_d / v_max
            constraint_violation = torch.relu(min_possible_time - y_pred)
            
            loss = base_mse + lambda_pen * torch.mean(constraint_violation)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        if (epoch + 1) % 15 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Total Loss: {epoch_loss/len(dataloader):.2f}")
            
    return model


In [40]:
def train_professional_model(X_scaled, y_train, distance_train):
    v_max = 15.0       
    lambda_pen = 15.0  # Slightly lower penalty to prioritize base learning
    epochs = 100       # Increased epochs because the LR scheduler will handle the finish
    batch_size = 32    # SMALLER BATCHES: Adds 'stochastic noise' to help find better minima
    
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    d_tensor = torch.tensor(distance_train, dtype=torch.float32).view(-1, 1)
    
    dataset = TensorDataset(X_tensor, y_tensor, d_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = PhysicsInformedNN(input_dim=X_scaled.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # NEW: Learning Rate Scheduler
    # It will reduce the LR by half if the loss doesn't improve for 5 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    print("Starting Professional Training Loop (Huber Loss + LR Scheduler)...\n")
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for batch_X, batch_y, batch_d in dataloader:
            optimizer.zero_grad()
            y_pred = model(batch_X)
            
            # UPGRADE: Huber Loss instead of MSE for outlier robustness
            base_loss = F.huber_loss(y_pred, batch_y, delta=1.0)
            
            # The KKT Physics Constraint
            min_possible_time = batch_d / v_max
            constraint_violation = torch.relu(min_possible_time - y_pred)
            
            # Generalized Lagrangian
            loss = base_loss + lambda_pen * torch.mean(constraint_violation)
            
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_epoch_loss = epoch_loss / len(dataloader)
        # Update the scheduler with the current loss
        scheduler.step(avg_epoch_loss)
        
        if (epoch + 1) % 20 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_epoch_loss:.4f} | LR: {current_lr}")
            
    return model

Custom Model Evaluation

In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("--- 1. Applying StandardScaler ---")
# We scale the features so the Neural Network gradients flow smoothly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# A smarter distance proxy: Assume the next stop is further away if the bus is moving fast.
# (If speed is 0, assume a baseline distance of 300m).
dynamic_distance_train = np.where(X_train['speed'] > 0, X_train['speed'] * 45, 300.0)
dynamic_distance_test = np.where(X_test['speed'] > 0, X_test['speed'] * 45, 300.0)


print("\n--- 2. Executing Tuned Training Loop ---")
tuned_model = train_tuned_model(X_train_scaled, y_train, dynamic_distance_train)

print("\n--- 3. Evaluating Tuned Constraint Engine ---")
tuned_model.eval()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    pytorch_preds = tuned_model(X_test_tensor).numpy()

pytorch_mae = mean_absolute_error(y_test, pytorch_preds)
pytorch_rmse = np.sqrt(mean_squared_error(y_test, pytorch_preds))

print("\n--- TUNED PHYSICS-INFORMED PYTORCH RESULTS ---")
print(f"Mean Absolute Error (MAE): {pytorch_mae:.2f} seconds")
print(f"Root Mean Squared Error (RMSE): {pytorch_rmse:.2f} seconds")

--- 1. Applying StandardScaler ---

--- 2. Executing Tuned Training Loop ---
Starting Training with Tuned Lambda: 10.0...

Epoch 15/200 | Total Loss: 80360.06
Epoch 30/200 | Total Loss: 75683.13
Epoch 45/200 | Total Loss: 69334.16
Epoch 60/200 | Total Loss: 64184.20
Epoch 75/200 | Total Loss: 60369.30
Epoch 90/200 | Total Loss: 57531.66
Epoch 105/200 | Total Loss: 55248.84
Epoch 120/200 | Total Loss: 53064.06
Epoch 135/200 | Total Loss: 51314.16
Epoch 150/200 | Total Loss: 49604.20
Epoch 165/200 | Total Loss: 47928.56
Epoch 180/200 | Total Loss: 46719.33
Epoch 195/200 | Total Loss: 46327.77

--- 3. Evaluating Tuned Constraint Engine ---

--- TUNED PHYSICS-INFORMED PYTORCH RESULTS ---
Mean Absolute Error (MAE): 106.09 seconds
Root Mean Squared Error (RMSE): 229.57 seconds
